# 05. Pipeline RAG Completo (Recuperação + Geração Fundamentada em Evidências)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:**
- Demonstração completa do fluxo: Pergunta -> Transformação -> Recuperação Semântica -> Contexto -> **Geração com Modelo de Linguagem** fundamentada em evidências.
- Respostas estritamente ancoradas nas avaliações reais dos clientes (Voice of Customer).
- Prevenção ativa de alucinações e tratamento de perguntas fora do escopo ou sem dados suficientes.

**Arquitetura de Geração Plugável (interface `GeradorResposta`):**
- **`local`** (padrão) — `SintetizadorLocalVoC`: síntese por regras, **100% offline, sem chaves e sem custo**.
- **`gemini`** — `GeradorGemini`: LLM de nuvem (Google Gemini) para geração em linguagem natural, **opt-in** via variável de ambiente `GEMINI_API_KEY`, com **degradação graciosa** para o gerador local em caso de indisponibilidade (sem SDK/chave/rede).

In [1]:
import sys
import os
sys.path.append('../')

from src.rag_pipeline import PipelineRAG, SintetizadorLocalVoC, GeradorGemini, DEFAULT_PROMPT_TEMPLATE
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

DB_PATH = '../data/vector_db/'

# Seleção do motor de geração ("G" do RAG).
# - 'local'  : 100% offline, por regras (padrão).
# - 'gemini' : LLM de nuvem; requer `pip install google-genai` e a variável de ambiente
#              GEMINI_API_KEY. Sem isso, o pipeline degrada automaticamente para o local.

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY não encontrada no .env')

PROVIDER = os.getenv('GEN_PROVIDER')
pipeline_rag = PipelineRAG(db_path=DB_PATH, provider=PROVIDER)

print(f"Pipeline RAG conectado ao ChromaDB local! Motor de geração: {type(pipeline_rag.gerador).__name__}")
print(f"Total de documentos indexados na base: {pipeline_rag.collection.count():,}")

# Diagnóstico do provider de nuvem (quando selecionado)
if isinstance(pipeline_rag.gerador, GeradorGemini):
    if pipeline_rag.gerador.disponivel:
        print(f"Gemini ativo: modelo '{pipeline_rag.gerador.model_name}'.")
    else:
        print(f"Gemini indisponível ({pipeline_rag.gerador.erro_init}). Usando fallback local automaticamente.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pipeline RAG conectado ao ChromaDB local! Motor de geração: GeradorGemini


Total de documentos indexados na base: 40,668
Gemini ativo: modelo 'gemini-3.8-flash'.


## 1. Engenharia de Prompt e Diretrizes de Fidelidade (Anti-Alucinação)
Visualizamos o template de prompt que orienta a síntese analítica e impõe a recusa caso as evidências não sustentem a resposta.

In [2]:
print("=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===")
print(DEFAULT_PROMPT_TEMPLATE)

=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===

Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações de e-commerce.
Sua missão é responder à pergunta do gestor ESTRITAMENTE com base nas Evidências Recuperadas abaixo.

EVIDÊNCIAS RECUPERADAS DA BASE DE CONHECIMENTO:
{context}

DIRETRIZES OBRIGATÓRIAS:
1. FIDELIDADE AOS DADOS: Responda apenas com fatos sustentados pelos comentários acima. Nunca invente ou assuma informações externas.
2. RASTREABILIDADE E EVIDÊNCIAS: Cite trechos ou menções dos comentários que fundamentam cada insight apresentado.
3. TRATAMENTO DE INSUFICIÊNCIA: Se as evidências recuperadas forem insuficientes ou a pergunta estiver fora do escopo da base de clientes, responda EXATAMENTE:
   "Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta."

Pergunta do Gestor: {question}

Resposta Fundamentada do Analista:



## 2. Cenário 1: Investigação de Reclamações de Entrega (Edital)
Pergunta de negócio: *"O que os clientes relatam sobre problemas relacionados à entrega?"*  
Utilizamos o **Filtro de Metadados** (`review_score <= 2`) para recuperar estritamente experiências insatisfatórias.

In [3]:
pergunta_1 = "O que os clientes relatam sobre problemas relacionados à entrega?"
resultado_1 = pipeline_rag.responder(
    pergunta=pergunta_1,
    n_results=4,
    where_filter={"review_score": {"$lte": 2}}
)

print(f"PERGUNTA: {resultado_1['pergunta']}\n")
print(resultado_1['resposta'])

PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?

Com base estritamente nas evidências recuperadas, os clientes relatam os seguintes pontos sobre problemas relacionados à entrega:

1. **Atrasos e Falta de Suporte/Tratamento:**
   * Há relatos de atraso no cumprimento dos prazos e ausência de suporte para solucionar o problema. Conforme apontado na **Evidência 1**, o cliente reclama de *"falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado"* e menciona que fez *"duas compras, uma com prazo de entrega e , as atrasou"*.

2. **Falta de Comunicação sobre Dificuldades:**
   * Os consumidores apontam que a loja falha em comunicar problemas operacionais ou imprevistos na entrega. Na **Evidência 4**, o cliente destaca que *"se houve alguma dificuldade para entrega do produto, no mínimo, o comprador deveria ser avisado e negociado uma nova situação de entrega"*.

3. **Incoerência Logística e Dificuldades Criad

## 3. Cenário 2: Busca por Experiências Positivas e Elogios (Edital)
Pergunta de negócio: *"Quais aspectos da experiência de compra são mais elogiados pelos clientes?"*  
Filtramos avaliações promotoras (`review_score >= 4`).

In [4]:
pergunta_2 = "Quais aspectos da experiência de compra são mais elogiados pelos clientes?"
resultado_2 = pipeline_rag.responder(
    pergunta=pergunta_2,
    n_results=4,
    where_filter={"review_score": {"$gte": 4}}
)

print(f"PERGUNTA: {resultado_2['pergunta']}\n")
print(resultado_2['resposta'])

PERGUNTA: Quais aspectos da experiência de compra são mais elogiados pelos clientes?

Com base nas avaliações analisadas, os principais aspectos da experiência de compra elogiados pelos clientes são:

1. **Agilidade na Entrega e Condição dos Produtos:**
   * Os clientes destacam a rapidez no envio e a integridade dos itens. Na **Evidência 2**, o consumidor elogia a entrega antecipada e a qualidade: *"o produto desta compra chegou antes do prazo maximo para entrega"* e relata que *"sempre recebi em perfeitas condições"*. O tema também é associado a uma experiência positiva na **Evidência 4**, cujo título é *"entrega"* e o comentário afirma: *"foi poditiva a experiência de compra"*.

2. **Qualidade do Atendimento e Compromisso do Vendedor:**
   * A conduta e o suporte da loja foram positivamente avaliados. Na **Evidência 2**, o vendedor é classificado como *"excelente vendedor"*. Já na **Evidência 3**, sob o título *"bom serviço"*, o cliente agradece nominalmente pelo suporte e postura d

## 4. Cenário 3: Tratamento de Perguntas sem Evidências Suficientes / Fora de Escopo
Conforme o Item 4 do edital: *"Soluções baseadas em IA Generativa podem produzir respostas aparentemente convincentes mesmo quando não existem informações suficientes para sustentá-las. Sua solução deverá considerar situações em que a pergunta esteja fora do escopo da base de conhecimento."*  

Testamos uma pergunta esdrúxula: *"Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"*.

In [5]:
pergunta_fora_escopo = "Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"
resultado_3 = pipeline_rag.responder(
    pergunta=pergunta_fora_escopo,
    n_results=3
)

print(f"PERGUNTA: {resultado_3['pergunta']}\n")
print(f"RESPOSTA DO SISTEMA:")
print(resultado_3['resposta'])

PERGUNTA: Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?

RESPOSTA DO SISTEMA:
Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta.


## 5. Rastreabilidade das Evidências (Auditoria Completa)
O sistema permite auditar detalhadamente todos os metadados, identificadores (`review_id`) e categorias retornadas pela base vetorial.

In [6]:
print("Auditoria dos IDs recuperados para a Pergunta 1:")
for i, (doc_id, meta) in enumerate(zip(resultado_1['ids'], resultado_1['metadados']), 1):
    print(f"[{i}] ID do Review: {doc_id} | Nota no BD: {meta.get('review_score')}")

Auditoria dos IDs recuperados para a Pergunta 1:
[1] ID do Review: 4baf65ba0f4dc753e9cf3d8863433da0 | Nota no BD: 2
[2] ID do Review: 12416af18a71403380ecd6e6ca53332c | Nota no BD: 1
[3] ID do Review: 66179b1ca078cdb08c6ca238a0a4c654 | Nota no BD: 1
[4] ID do Review: c20dc8f95c7e0a17a4192984287a48ef | Nota no BD: 1
